In [1]:
import pandas as pd
f1 = pd.read_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/HTO/qc_meta.csv")
f2 = pd.read_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/SNP/qc_meta.csv")
# # Ensure df1 has 'barcode' as a column
# df1 = df1.reset_index().rename(columns={"index": "barcode"})

f1.loc[f1["DoubletStatus"] == "Doublet", "Donor"] = "Doublet"

mergedf = f1.merge(f2[["barcode", "donor_id"]], on="barcode", how="left")

In [2]:
total_counts_donor_id = mergedf["donor_id"].value_counts().reset_index()
total_counts_donor_id.columns = ["donor_id", "Total"]
print("\nTotal counts for each donor_id:")
print(total_counts_donor_id.sort_values(by="donor_id").to_string(index=False))

# 3. Compute total per Donor
total_counts_donor = mergedf["Donor"].value_counts().reset_index()
total_counts_donor.columns = ["Donor", "Total"]
# print("\nTotal counts for each Donor:")
# print(total_counts_donor.sort_values(by="Donor").to_string(index=False))

donor_counts_qc = mergedf.groupby(["donor_id", "Donor"]).size().reset_index(name="count")
# print(donor_counts_qc.sort_values(by=["donor_id", "count"], ascending=[True, False]).to_string(index=False))



Total counts for each donor_id:
  donor_id  Total
    donor0    700
    donor1    740
    donor2    652
    donor3    786
    donor4    872
    donor5   1037
    donor6    717
    donor7    955
    donor8   1064
unassigned     90


In [3]:
merged = pd.read_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/both/qc_meta.csv")
merged.loc[merged["DoubletStatus"] == "Doublet", "Donor"] = "Doublet"
donor_counts_qc = merged.groupby(["donor_id", "Donor"]).size().reset_index(name="count")
# print(donor_counts_qc.sort_values(by=["donor_id", "count"], ascending=[True, False]).to_string(index=False))



In [4]:
Donor_counts_qc = merged.groupby(["Donor","donor_id"]).size().reset_index(name="count")
# print(Donor_counts_qc.sort_values(by=["Donor", "count"], ascending=[True, False]).to_string(index=False))

In [5]:
# 2. Compute total per donor_id
total_counts_donor_id = merged["donor_id"].value_counts().reset_index()
total_counts_donor_id.columns = ["donor_id", "Total"]
# print("\nTotal counts for each donor_id:")
# print(total_counts_donor_id.sort_values(by="donor_id").to_string(index=False))

# 3. Compute total per Donor
total_counts_donor = merged["Donor"].value_counts().reset_index()
total_counts_donor.columns = ["Donor", "Total"]
# print("\nTotal counts for each Donor:")
# print(total_counts_donor.sort_values(by="Donor").to_string(index=False))

In [6]:
df1 = pd.read_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/both/qc_meta.csv")
df1= df1[["barcode","Donor","donor_id","DoubletStatus","prob_max","prob_doublet"]]
df2 = pd.read_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/both/qc_SNP_HTO.csv")
df = pd.merge(df1, df2, on="barcode")

In [7]:
# df["donor_id"] = df["donor_id"].astype(str).str.strip()
# df["Donor"] = df["Donor"].astype(str).str.strip()

# # # Remove rows where both donor_id and Donor are exactly "Doublet"
# # df = df[~((df["donor_id"] == "doublet") & (df["Donor"] == "Doublet"))]
df.loc[df["DoubletStatus"] == "Doublet", "Donor"] = "Doublet"
df.loc[df["donor_id"] == "doublet", "donor_id"] = "Doublet"


In [8]:
donor_counts = df.groupby(["donor_id", "Donor"]).size().reset_index(name="count")

# Find the Donor with the max count for each donor_id
max_donor = donor_counts.loc[donor_counts.groupby('donor_id')['count'].idxmax()]

# Create a mapping of donor_id to the Donor with the highest count
donor_id_to_max_donor = max_donor.set_index('donor_id')['Donor'].to_dict()

# Create a mapping of donor_id to the max count
donor_id_to_max_count = max_donor.set_index('donor_id')['count'].to_dict()

# Function to assign the final donor based on maximum count match
def assign_final_donor(row):
    donor_id = row["donor_id"]
    
    # Keep "Doublet" and "Unassigned" as-is
    if donor_id in ["Doublet", "Unassigned"]:
        return donor_id

    max_donor_for_id = donor_id_to_max_donor.get(donor_id)
    max_count_for_id = donor_id_to_max_count.get(donor_id)

    current_donor_count = donor_counts[
        (donor_counts["donor_id"] == donor_id) & 
        (donor_counts["Donor"] == row["Donor"])
    ]["count"].values

    if current_donor_count.size > 0 and current_donor_count[0] == max_count_for_id:
        return max_donor_for_id
    else:
        return donor_id


# Apply the function to assign the final donor
df["SNP_donor"] = df.apply(assign_final_donor, axis=1)

In [20]:
# Your custom donor ID to letter mapping
custom_label_map = {
    'B10148': 'Donor A',
    'B30477': 'Donor B',
    'B30864': 'Donor C',
    'B10678': 'Donor D',
    'B30248': 'Donor E',
    'B10237': 'Donor F',
    'B40448': 'Donor G',
    'B30803': 'Donor H',
    'B30796': 'Donor I'
}

# Print with custom labels
for k, v in donor_id_to_max_donor.items():
    label = custom_label_map.get(v, v)  # fallback to original if not found
    print(f"donor_id '{k}' is most likely {label}")


donor_id 'Doublet' is most likely Donor F
donor_id 'donor0' is most likely Donor G
donor_id 'donor1' is most likely Donor A
donor_id 'donor2' is most likely Donor F
donor_id 'donor3' is most likely Donor E
donor_id 'donor4' is most likely Donor I
donor_id 'donor5' is most likely Donor H
donor_id 'donor6' is most likely Donor D
donor_id 'donor7' is most likely Donor C
donor_id 'donor8' is most likely Donor B
donor_id 'unassigned' is most likely Doublet


In [21]:
#df.to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/both/ds2.donor.csv", index=False)

In [25]:
hto_columns = ["B30803", "B10148", "B40448", "B20340", "B30477", "B10678", "B30864", "B30248", "B30796", "B10237"]

# Ensure all HTO columns are numeric
df[hto_columns] = df[hto_columns].apply(pd.to_numeric, errors='coerce')

df["best_hto"] = df[hto_columns].idxmax(axis=1)


In [26]:
def assign_final_donor(row):
    # Keep Doublet and Unassigned unchanged
    if row["SNP_donor"] in ["Doublet", "Unassigned"]:
        return row["SNP_donor"]
    
    # If prob_max is high and all labels agree, assign the confident Donor
    if row["prob_max"] >= 0.9 and row["Donor"] == row["best_hto"] == row["SNP_donor"]:
        return row["Donor"]
    
    # Otherwise fallback to SNP_donor
    return row["SNP_donor"]

df["fin_donor"] = df.apply(assign_final_donor, axis=1)

In [27]:
df.to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/both/step1.csv", index=False)


In [28]:
 filtered_df = df[df["fin_donor"].isin(df["donor_id"].unique())].copy()


In [29]:
def resolve_doublets(row, hto_columns, donor_id_to_max_donor, threshold=0.58):
    donor_id = row["donor_id"]
    current_donor = row["Donor"]
    fin_donor = row["fin_donor"]
    
    if fin_donor in ["Doublet", "Unassigned"] or current_donor != "Doublet":
        return row["fin_donor"]

    hto_values = row[hto_columns]
    sorted_hto = hto_values.sort_values(ascending=False)

    top1_name = sorted_hto.index[0].replace("HTO_", "")
    top1_val = sorted_hto.iloc[0]
    top2_val = sorted_hto.iloc[1]

    true_donor = donor_id_to_max_donor.get(donor_id)

    # Use ratio of second strongest to strongest HTO
    if (top2_val / top1_val) > threshold:
        return "Actual_Doublet"
    elif (top2_val / top1_val) < threshold:  # Optional: use absolute threshold for top1
         return true_donor
    else:
        return donor_id



In [30]:
filtered_df["f_donor"] = df.apply(
    resolve_doublets,
    axis=1,
    args=(hto_columns, donor_id_to_max_donor, 0.58)  # 0.2 is threshold, adjust if needed
)

In [31]:
filtered_df.to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/both/step_2.csv", index=False)

In [32]:
filter_df = filtered_df[filtered_df["f_donor"].isin(df["donor_id"].unique())].copy()
filter_df.to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/both/rest.csv", index=False)

In [33]:

# Assuming filtered_df is your existing DataFrame
# Modify this condition as needed if you have specific criteria for "negative"
def update_f_donor(row):
    if row['Donor'] == 'Negative' and row['donor_id'] == 'unassigned':
        row['f_donor'] = 'Negative'
    return row

# Apply the function to update the 'f_donor' column based on the condition
filter_df = filter_df.apply(update_f_donor, axis=1)

# Check the result
# print(filter_df[['Donor', 'donor_id', 'f_donor']])


In [34]:
filter_df.to_csv("/media/igl2/828f6e4d-cda7-48c0-99c1-478e80aeecda/Shubham/DS2/both/rest.csv", index=False)

In [35]:
def resolve_doublets(row, hto_columns, donor_id_to_max_donor, threshold=0.58):
    donor_id = row["donor_id"]
    current_donor = row["Donor"]
    fin_donor = row["fin_donor"]

    # Case: donor_id is unassigned and current Donor is Doublet
    if donor_id == "unassigned" and current_donor == "Doublet":
        hto_values = row[hto_columns]
        sorted_hto = hto_values.sort_values(ascending=False)

        top1_val = sorted_hto.iloc[0]
        top2_val = sorted_hto.iloc[1]

        ratio = top2_val / top1_val if top1_val != 0 else 0

        if ratio > threshold:
            return "Actual_Doublet"
        else:
            return "Negative"
    
    # If not matching the above condition, return the current fin_donor value
    return fin_donor


In [36]:
filter_df["fa_donor"] = filter_df.apply(
    resolve_doublets,
    axis=1,
    args=(hto_columns, donor_id_to_max_donor, 0.58)  # 0.2 is threshold, adjust if needed
)

In [37]:
def resolve_doublets(row, hto_columns, donor_id_to_max_donor, threshold=0.58):
    donor_id = row["donor_id"]
    current_donor = row["Donor"]
    fin_donor = row["fa_donor"]

    # Case: donor_id is unassigned and current Donor is Doublet
    if donor_id == "Doublet" and current_donor == "Donor":
        hto_values = row[hto_columns]
        sorted_hto = hto_values.sort_va
        lues(ascending=False)

        top1_val = sorted_hto.iloc[0]
        top2_val = sorted_hto.iloc[1]

        ratio = top2_val / top1_val if top1_val != 0 else 0

        if ratio > threshold:
            return "Actual_Doublet"
        else:
            return "Negative"
    
    # If not matching the above condition, return the current fin_donor value
    return fin_donor


In [38]:
filter_df["fam_donor"] = filter_df.apply(
    resolve_doublets,
    axis=1,
    args=(hto_columns, donor_id_to_max_donor, 0.58)  # 0.2 is threshold, adjust if needed
)